In [ ]:
import os, time, requests, jieba
from typing import List
import chromadb

# 引入 LlamaIndex 核心元件
from llama_index.core import StorageContext, Settings, VectorStoreIndex
from llama_index.core.prompts.prompts import SimpleInputPrompt
from llama_index.core.schema import Document

# 引入 ChromaDB 整合元件
from llama_index.vector_stores.chroma import ChromaVectorStore

# 引入 LlamaIndex 的 Ollama 整合
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

class LlamaIndexRAG:    
    """    支援動態參數配置並嚴格按頁分片的 ChromaDB RAG 系統    """
    def __init__(self, model_name: str, embed_model_name: str, base_url: str = "http://localhost:11434"):
        self.model_name, self.embed_model_name, self.base_url, self.index= model_name,embed_model_name, base_url, None

        self._check_ollama_status()

        # 配置 LLM
        self.llm = Ollama( model=self.model_name, base_url=self.base_url, 
            request_timeout=30000,
            additional_kwargs={"num_ctx": 8192, "num_predict": 1024}       )
        
        # 配置 Embedding
        self.embed_model = OllamaEmbedding(model_name=self.embed_model_name, base_url=self.base_url,
            request_timeout=30000, additional_kwargs={"num_ctx": 4096, "truncate": True}        )

        Settings.llm, Settings.embed_model = self.llm, self.embed_model
        Settings.chunk_size, Settings.chunk_overlap = 4096, 0

        self.qa_template = SimpleInputPrompt("""
        你是一個精準的招生資料分析助手。請根據提供的參考資料回答用戶問題。

        【參考資料】
        {context_str}

        【問題】
        {query_str}

        【指令】
        1. 若資料包含答案（如表格中的考試科目、名額），請以繁體中文清楚列出。
        2. 保持客觀，優先使用資料內的用語。
        3. 如果資料中確實沒有相關資訊，請回答：我不知道。
        4. 忽略分頁標記等雜訊。
        """)

    def _check_ollama_status(self):
        try:
            response = requests.get(f"{self.base_url}/api/tags")
            if response.status_code == 200:
                print(f"✅ Ollama 伺服器連接成功。")
        except:
            raise ConnectionError(f"無法連接到 Ollama。")

    def create_index_with_chroma(self, documents: List[str], db_path: str, collection_name: str = "admission_guide"):
        """
        手動處理 documents，確保每一頁是一個節點，不進行二次切片
        """
        if not os.path.exists(db_path):
            os.makedirs(db_path)

        db = chromadb.PersistentClient(path=db_path)
        chroma_collection = db.get_or_create_collection(collection_name)
        vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
        storage_context = StorageContext.from_defaults(vector_store=vector_store)

        if chroma_collection.count() > 0:
            print(f"🔍 偵測到現有 Vector DB ({db_path})，共有 {chroma_collection.count()} 頁資料，直接載入...")
            self.index = VectorStoreIndex.from_vector_store(vector_store, storage_context=storage_context)
        else:
            print("📝 Vector DB 為空，正在按頁面建立索引 (不進行二次切片)...")
            start_time = time.time()
            
            # 關鍵修改：將每一頁包裝成 Document，並直接作為 nodes 傳入
            # 這樣就不會觸發 SentenceSplitter，保留表格完整性
            nodes = [Document(text=page_content) for page_content in documents]
            
            self.index = VectorStoreIndex(nodes, storage_context=storage_context)
            print(f"✅ 索引建立完成。共處理 {len(nodes)} 頁。耗時：{time.time() - start_time:.2f} 秒。")

    def rag_query(self, query: str, top_k: int) -> str:
        if self.index is None: return "錯誤：索引尚未建立。"
        query_engine = self.index.as_query_engine(
            text_qa_template=self.qa_template, 
            similarity_top_k=top_k
        )
        response = query_engine.query(query)
        # --- DEBUG 輸出邏輯 ---
        print(f"\n[DEBUG] 檢索到 {len(response.source_nodes)} 個相關分片：")
        for i, node in enumerate(response.source_nodes):
            # 取得原始文字並過濾換行符號方便閱讀
            content_preview = node.node.get_content().replace('\n', ' ')[:40]
            score = getattr(node, 'score', 'N/A')
            print(f"  分片 {i+1} (相似度: {score:.4f}): {content_preview}...")
        print("-" * 30)
        # ---------------------
        return response.response

def main(data_file, db_path, model_name="llama3:8b", emodel_name="mxbai-embed-large", top_k=3):
    print(f"🚀 啟動 RAG 系統 (按頁分片模式)")
    print(f"   - LLM: {model_name} | Embedding: {emodel_name} | Top_K: {top_k}")
    print(f"   - 數據源: {data_file}\n" + "="*50)

    rag = LlamaIndexRAG(model_name=model_name, embed_model_name=emodel_name)

    # 讀取檔案並按照「=== 第」分頁
    if not os.path.exists(data_file):
        print(f"❌ 找不到文件 '{data_file}'")
        return
        
    with open(data_file, "r", encoding="utf-8") as f:
        content = f.read()
    
    # 嚴格遵循您提供的分頁邏輯
    pages = content.split("=== 第 ")
    pages = pages[1:]
    documents = ["=== 第 " + p.strip() for p in pages]
    
    print(f"📄 共讀入 {len(documents)} 頁。")
    if documents:
        print(f"📸 第一頁預覽: {documents[0][:30]}...")

    # 建立索引
    rag.create_index_with_chroma(documents, db_path=db_path)

    print("\n🔍 系統已就緒 (輸入 quit 結束)")
    while True:
        query = input("❓ 問題: ").strip()
        if query.lower() == "quit": break
        answer = rag.rag_query(query, top_k=top_k)
        print(f"🤖 回答: {answer}\n", "-" * 50)

if __name__ == "__main__":
    main(data_file="114_碩士班招生簡章_分頁.txt",
         db_path="chroma_storage_v1", 
         model_name="gpt-oss:20b", 
         emodel_name="mxbai-embed-large",
         top_k=10  
    )

🚀 啟動 RAG 系統 (按頁分片模式)
   - LLM: gpt-oss:20b | Embedding: mxbai-embed-large | Top_K: 10
   - 數據源: 114_碩士班招生簡章_分頁.txt
✅ Ollama 伺服器連接成功。
📄 共讀入 49 頁。
📸 第一頁預覽: === 第 1 頁 ===
113 年 9 月 19 日本校...
🔍 偵測到現有 Vector DB (chroma_storage_v1)，共有 49 頁資料，直接載入...

🔍 系統已就緒 (輸入 quit 結束)


❓ 問題:  企業管理學系招生名額?



[DEBUG] 檢索到 10 個相關分片：
  分片 1 (相似度: 0.6600 if score else 'N/A'): === 第 3 頁 === 報名流程圖    1.列印出之報名表請仔細核對並由本...
  分片 2 (相似度: 0.6551 if score else 'N/A'): === 第 46 頁 === 考 生 姓 名   系（所）組          ...
  分片 3 (相似度: 0.6536 if score else 'N/A'): === 第 10 頁 === 5  十一、本校註冊入學（含在學及休學）或獲准保留...
  分片 4 (相似度: 0.6442 if score else 'N/A'): === 第 7 頁 === 2  1.凡報考本校「114 學年度碩士班招生」考試...
  分片 5 (相似度: 0.6442 if score else 'N/A'): === 第 42 頁 === 世新大學 114 學年度碩士班招生        ...
  分片 6 (相似度: 0.6385 if score else 'N/A'): === 第 43 頁 === 世新大學 114 學年度碩士班招生        ...
  分片 7 (相似度: 0.6358 if score else 'N/A'): === 第 25 頁 === 20  拾壹、網路查詢服務（成績、榜單、報到狀況）...
  分片 8 (相似度: 0.6212 if score else 'N/A'): === 第 45 頁 === 世新大學 114 學年度碩士班招生  報名考生學生...
  分片 9 (相似度: 0.6122 if score else 'N/A'): === 第 6 頁 === 1  世新大學 114 學年度碩士班招生簡章  壹、...
  分片 10 (相似度: 0.6065 if score else 'N/A'): === 第 5 頁 === 附錄一：入學大學同等學力認定標準（條文摘錄）  附錄...
------------------------------
🤖 回答: 我不知道。
 --------------------------------------------------
